In [4]:
# import paramiko
# import time
# import re
# import pandas as pd
# from pathlib import Path
# from datetime import datetime

# # ---- CM Connection Credentials ----
# SAT_HOST = "172.16.70.39"
# SAT_PORT = 5022
# SAT_USERNAME = "dadmin"
# SAT_PASSWORD = "Avaya@123"

# # ---- Locate latest list_trunk_group Excel ----
# reports_dir = Path("/Users/sooryaraysam/Checklist Automation/Trunks/reports")
# files = sorted(
#     reports_dir.glob("list_trunk-group*.xlsx"),
#     key=lambda f: f.stat().st_mtime,
#     reverse=True
# )

# if not files:
#     raise FileNotFoundError("❌ No file starting with 'list_trunk_group' found in reports/")

# list_file = files[0]
# print(f"📘 Using trunk list: {list_file}")

# # ---- Read Grp No column ----
# df_trunks = pd.read_excel(list_file)

# if "Grp No" not in df_trunks.columns:
#     raise ValueError("❌ Column 'Grp No' not found in trunk list file")

# # Clean and filter numeric trunk numbers
# valid_trunks = []
# for val in df_trunks["Grp No"].dropna():
#     try:
#         num = int(str(val).strip())
#         valid_trunks.append(num)
#     except ValueError:
#         print(f"⚠️ Skipping non-numeric entry in 'Grp No': {val}")

# trunk_numbers = sorted(set(valid_trunks))
# print(f"🔢 Found {len(trunk_numbers)} valid trunk groups to test: {trunk_numbers}")


# # ---- Connect to CM ----
# ssh = paramiko.SSHClient()
# ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
# ssh.connect(SAT_HOST, port=SAT_PORT, username=SAT_USERNAME, password=SAT_PASSWORD, look_for_keys=False)
# channel = ssh.invoke_shell()
# time.sleep(1)
# channel.recv(9999)

# # ---- Regex for test trunk table rows ----
# row_pattern = re.compile(
#     r"^(\S+)\s+(\S+)\s+(\S+)\s+(\d+)\s+(\S+)(?:\s+(\S+))?$"
# )

# all_results = []

# # ---- Loop through each trunk ----
# for trunk_no in trunk_numbers:
#     print(f"\n🚀 Testing trunk {trunk_no} ...")
#     channel.send(f"test trunk {trunk_no}\n")
#     time.sleep(2)
#     page = 1

#     while True:
#         output = channel.recv(99999).decode(errors="ignore")
#         lines = output.splitlines()
#         data_lines = []

#         for line in lines:
#             m = row_pattern.match(line.strip())
#             if m:
#                 port, mtce, alt, test_no, result, err_code = m.groups()
#                 data_lines.append({
#                     "Trunk No": trunk_no,
#                     "Port": port,
#                     "Mtce Name": mtce,
#                     "Alt. Name": alt,
#                     "Test No.": test_no,
#                     "Result": result,
#                     "Error Code": err_code if err_code else ""
#                 })

#         if not data_lines:
#             print(f"🛑 No data on page {page} for trunk {trunk_no}, moving on.")
#             break

#         all_results.extend(data_lines)
#         print(f"✅ Page {page} parsed ({len(data_lines)} rows)")

#         # Send F7 for next page
#         channel.send("\x1b[18~")
#         page += 1
#         time.sleep(1.5)

# # ---- Save all results ----
# if all_results:
#     out_df = pd.DataFrame(all_results)
#     timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#     output_file = reports_dir / f"test_trunk_results_{timestamp}.xlsx"
#     out_df.to_excel(output_file, index=False)
#     print(f"\n✅ All trunk test results saved to: {output_file}")
# else:
#     print("\n⚠️ No test trunk data parsed from any group.")

# # ---- Cleanup ----
# channel.close()
# ssh.close()


📘 Using trunk list: /Users/sooryaraysam/Checklist Automation/Trunks/reports/list_trunk-group_20251023_182057.xlsx
⚠️ Skipping non-numeric entry in 'Grp No': Command successfully completed: list trunk-group
🔢 Found 3 valid trunk groups to test: [1, 3, 4]

🚀 Testing trunk 1 ...
🛑 No data on page 1 for trunk 1, moving on.

🚀 Testing trunk 3 ...
🛑 No data on page 1 for trunk 3, moving on.

🚀 Testing trunk 4 ...
🛑 No data on page 1 for trunk 4, moving on.

⚠️ No test trunk data parsed from any group.


## Test TRUNK by referring to list group trunk

In [11]:
import paramiko
import time
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
import os
import sys

# ---- CM Connection Credentials ----
SAT_HOST = "172.16.70.39"
SAT_PORT = 5022
SAT_USERNAME = "dadmin"
SAT_PASSWORD = "Avaya@123"

# ---- Ensure absolute reports folder path ----
try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    script_dir = Path(os.getcwd()).resolve()

reports_dir = script_dir / "reports"
reports_dir.mkdir(exist_ok=True)

# ---- Locate latest list_trunk-group Excel ----
files = sorted(
    reports_dir.glob("list_trunk-group*.xlsx"),
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

if not files:
    raise FileNotFoundError("❌ No file starting with 'list_trunk-group' found in reports/")

list_file = files[0]
print(f"📘 Using trunk list: {list_file}")

# ---- Read and clean trunk numbers ----
df_trunks = pd.read_excel(list_file)

if "Grp No" not in df_trunks.columns:
    raise ValueError("❌ Column 'Grp No' not found in trunk list file")

valid_trunks = []
for val in df_trunks["Grp No"].dropna():
    try:
        num = int(str(val).strip())
        valid_trunks.append(num)
    except ValueError:
        print(f"⚠️ Skipping non-numeric entry in 'Grp No': {val}")

trunk_numbers = sorted(set(valid_trunks))
print(f"🔢 Found {len(trunk_numbers)} valid trunk groups to test: {trunk_numbers}")

# ---- Connect to CM ----
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(
    SAT_HOST,
    port=SAT_PORT,
    username=SAT_USERNAME,
    password=SAT_PASSWORD,
    look_for_keys=False
)
channel = ssh.invoke_shell()
time.sleep(1)
channel.recv(9999)

# ---- Regex pattern for test trunk data ----
row_pattern = re.compile(r"^(\S+)\s+(\S+)\s+(\S+)\s+(\d+)\s+(\S+)(?:\s+(\S+))?$")

# ---- Prepare Excel output path ----
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = reports_dir / f"test_trunk_results_{timestamp}.xlsx"
print("\n📂 Absolute save path:", output_file.resolve())

# ---- Initialize DataFrame for cumulative results ----
columns = ["Trunk No", "Port", "Mtce Name", "Alt. Name", "Test No.", "Result", "Error Code"]
all_results_df = pd.DataFrame(columns=columns)

# ---- Run test trunk for each trunk group ----
for idx, trunk_no in enumerate(trunk_numbers, start=1):
    print(f"\n🚀 ({idx}/{len(trunk_numbers)}) Running test trunk {trunk_no} ...")
    channel.send(f"test trunk {trunk_no}\n")
    time.sleep(2)
    page = 1
    trunk_results = []

    while True:
        output = channel.recv(99999).decode(errors="ignore")
        lines = output.splitlines()
        page_data = []

        for line in lines:
            m = row_pattern.match(line.strip())
            if m:
                port, mtce, alt, test_no, result, err_code = m.groups()
                page_data.append({
                    "Trunk No": trunk_no,
                    "Port": port,
                    "Mtce Name": mtce,
                    "Alt. Name": alt,
                    "Test No.": test_no,
                    "Result": result,
                    "Error Code": err_code if err_code else ""
                })

        if not page_data:
            print(f"🛑 No data found on page {page} for trunk {trunk_no}")
            break

        trunk_results.extend(page_data)
        print(f"✅ Parsed page {page} for trunk {trunk_no} ({len(page_data)} rows)")

        # Next page (F7)
        channel.send("\x1b[18~")
        page += 1
        time.sleep(1.5)

    # ---- Write results for this trunk immediately ----
    if trunk_results:
        temp_df = pd.DataFrame(trunk_results)
        all_results_df = pd.concat([all_results_df, temp_df], ignore_index=True)
        all_results_df.to_excel(output_file, index=False)
        print(f"💾 Data for trunk {trunk_no} written to {output_file.name}")
    else:
        print(f"⚠️ No valid test data for trunk {trunk_no}")

# ---- Always create Excel (even if empty) with proper columns ----
if all_results_df.empty:
    print("⚠️ No data found for any trunks — creating blank Excel with standard columns.")
    blank_df = pd.DataFrame(columns=columns)
    blank_df.to_excel(output_file, index=False)

# ---- Cleanup ----
channel.close()
ssh.close()

print(f"\n✅ All test trunk results saved successfully to:\n{output_file.resolve()}")


📘 Using trunk list: /Users/sooryaraysam/Checklist Automation/Trunks/reports/list_trunk-group_20251023_182057.xlsx
⚠️ Skipping non-numeric entry in 'Grp No': Command successfully completed: list trunk-group
🔢 Found 3 valid trunk groups to test: [1, 3, 4]

📂 Absolute save path: /Users/sooryaraysam/Checklist Automation/Trunks/reports/test_trunk_results_20251024_121412.xlsx

🚀 (1/3) Running test trunk 1 ...
🛑 No data found on page 1 for trunk 1
⚠️ No valid test data for trunk 1

🚀 (2/3) Running test trunk 3 ...
🛑 No data found on page 1 for trunk 3
⚠️ No valid test data for trunk 3

🚀 (3/3) Running test trunk 4 ...
🛑 No data found on page 1 for trunk 4
⚠️ No valid test data for trunk 4
⚠️ No data found for any trunks — creating blank Excel with standard columns.

✅ All test trunk results saved successfully to:
/Users/sooryaraysam/Checklist Automation/Trunks/reports/test_trunk_results_20251024_121412.xlsx
